# M1 双卡 DDP 批处理（v11）：torchrun×2，bs32/rank，accum1 → eff-bs64，同 v9 实验语义

In [ ]:
# ===== v11 双卡DDP批处理：torchrun×2 + bs32/rank + accum1 = eff64，同 v9 实验，预期 ~4.8h =====
# 设计原则：训练是前台进程，任何崩溃的 traceback 直接出现在批处理日志里，无静默失败。
# 无隧道/sshd/watchdog——监控走 ntfy 信标（topic smolvla-ep-b7Vq2mN4tX）。
import os, re, glob, shutil, time, subprocess, urllib.request

W = "/kaggle/working"
TOPIC = "smolvla-ep-b7Vq2mN4tX"
CODE_SHA = "146d8474932c8b566717eb2738ba459337ab71c4"
REV = "a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4"
HUB_SHA = "5e8d12a6e2975b0e5e5fce7c8caf47c371d257b6"
os.environ["HF_HOME"] = W + "/hf_cache"
os.makedirs(W + "/logs", exist_ok=True)

def run(cmd, timeout=1800):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()

def beacon(msg):
    try:
        urllib.request.urlopen(urllib.request.Request("https://ntfy.sh/" + TOPIC, data=str(msg).encode()), timeout=8)
    except Exception as e:
        print("[beacon fail]", repr(e), flush=True)

# ---- 1) lerobot 安装（固定 commit）+ dtype 补丁 ----
def have_lerobot():
    return "0.6.2" in run("python -c 'import lerobot; print(lerobot.__version__)'")
if not have_lerobot():
    for cmd in ('uv pip install --system "lerobot[smolvla,libero] @ git+https://github.com/huggingface/lerobot.git@' + CODE_SHA + '"',
                'pip install --no-input "lerobot[smolvla,libero] @ git+https://github.com/huggingface/lerobot.git@' + CODE_SHA + '"'):
        print("[1] install ...", flush=True)
        print(run(cmd, timeout=1200)[-600:], flush=True)
        if have_lerobot():
            break
    if not have_lerobot():
        beacon("FAIL-install")
        raise RuntimeError("lerobot 安装失败（两种方式均失败，输出见上）")
run('sed -i \'s/torch_dtype="bfloat16"/torch_dtype="float32"/\' /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla/smolvlm_with_expert.py')
ok = "float32" in run("grep torch_dtype /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla/smolvlm_with_expert.py")
print("[1] lerobot 0.6.2 + dtype patch:", ok, flush=True)
if not ok:
    beacon("FAIL-dtype")
    raise RuntimeError("dtype 补丁失败")
beacon("OK-install")

# ---- 2) fp32 权重（model.safetensors 存在则跳过）----
CAST = """import torch, os
from lerobot.policies.smolvla import SmolVLAPolicy
p = SmolVLAPolicy.from_pretrained('lerobot/smolvla_base')
p.to(torch.float32)
p.save_pretrained('/kaggle/working/smolvla_base_fp32')
print('cast done')
"""
if not os.path.exists(W + "/smolvla_base_fp32/model.safetensors"):
    open("/tmp/cast.py", "w").write(CAST)
    out = run("python /tmp/cast.py", timeout=1200)
    print("[2] fp32 cast:", out[-120:], flush=True)
    if not os.path.exists(W + "/smolvla_base_fp32/model.safetensors"):
        beacon("FAIL-cast")
        raise RuntimeError("fp32 cast 失败: " + out[-400:])
else:
    print("[2] fp32 权重已存在，跳过", flush=True)

# ---- 3) processor 配置：无条件从 Hub 锁定版本拉取（不依赖本地缓存）----
from huggingface_hub import hf_hub_download
for fn in ("policy_preprocessor.json",
           "policy_preprocessor_step_5_normalizer_processor.safetensors",
           "policy_postprocessor.json",
           "policy_postprocessor_step_0_unnormalizer_processor.safetensors"):
    shutil.copy(hf_hub_download("lerobot/smolvla_base", fn, revision=HUB_SHA), W + "/smolvla_base_fp32/")
if not os.path.exists(W + "/smolvla_base_fp32/policy_preprocessor.json"):
    beacon("FAIL-processor")
    raise RuntimeError("processor 配置缺失 policy_preprocessor.json")
print("[3] processor 配置就绪（Hub sha " + HUB_SHA[:8] + "）", flush=True)
beacon("OK-weights")

# ---- 4) assets + libero 配置（失败仅影响训练内评测，不致命）----
ASSETS = """import time
from pathlib import Path
from huggingface_hub import snapshot_download
import importlib.util
home = Path.home()
t0 = time.time()
snapshot_download(repo_id='lerobot/libero-assets', repo_type='dataset', local_dir='/kaggle/working/libero_assets')
spec = importlib.util.find_spec('libero')
lib_dir = Path(spec.origin).parent / 'libero'
(home / '.libero').mkdir(exist_ok=True)
lines = [
    'assets: /kaggle/working/libero_assets',
    'bddl_files: %s/bddl_files' % lib_dir,
    'datasets: %s/../datasets' % lib_dir.parent,
    'init_states: %s/init_files' % lib_dir,
]
(home / '.libero' / 'config.yaml').write_text(chr(10).join(lines) + chr(10))
print('assets %.0fs' % (time.time() - t0))
"""
open("/tmp/assets.py", "w").write(ASSETS)
out = run("python /tmp/assets.py", timeout=1800)
print("[4] assets:", out[-150:], flush=True)
if "assets" not in out:
    print("[4][warn] 资产失败，训练内评测会失败", flush=True)

# ---- 5) 数据集缓存修复（pinned revision，3 次重试）----
open("/tmp/dsfix.py", "w").write(
    "from huggingface_hub import snapshot_download\n"
    "snapshot_download('lerobot/libero', repo_type='dataset', revision='" + REV + "', max_workers=4)\n"
    "print('DS_OK')\n")
for attempt in range(3):
    out = run("python /tmp/dsfix.py 2>&1 | tail -2", timeout=2400)
    if "DS_OK" in out:
        break
    print("[5] 数据集修复重试", attempt + 1, ":", out[-200:], flush=True)
    time.sleep(20)
else:
    beacon("FAIL-dataset")
    raise RuntimeError("数据集缓存修复失败: " + out[-300:])
print("[5] 数据集缓存完整（revision " + REV[:8] + "）", flush=True)
beacon("OK-setup")

# ---- 6) 磁盘保护：只保留最新 2 个 checkpoint ----
open(W + "/prune.sh", "w").write(
    '#!/bin/bash\nD=' + W + '/outputs/m1_base_libero/checkpoints\n'
    'while true; do sleep 300; [ -d "$D" ] || continue; '
    'ls "$D" | grep -E "^[0-9]+$" | sort -n | head -n -2 | while read c; do rm -rf "$D/$c"; done; done\n')
subprocess.Popen("bash " + W + "/prune.sh", shell=True, start_new_session=True)

# ---- 7) 心跳：每 10 分钟推送进度/GPU/磁盘 ----
HB = ('#!/bin/bash\nexport PATH=/opt/bin:$PATH\nexport LD_LIBRARY_PATH=/usr/local/nvidia/lib64:$LD_LIBRARY_PATH\n'
      'while true; do sleep 600\n'
      'S=$(tail -c 400 ' + W + '/logs/train.log | tr "\\r" "\\n" | grep "Training:" | tail -1)\n'
      'G=$(nvidia-smi --query-gpu=utilization.gpu --format=csv,noheader 2>/dev/null | head -1)\n'
      'D=$(df -BG /kaggle/working | tail -1 | awk "{print \\$4}")\n'
      'curl -s -m 8 -d "HB $S | gpu=$G | disk_free=$D" https://ntfy.sh/' + TOPIC + ' >/dev/null\ndone\n')
open(W + "/hb.sh", "w").write(HB)
subprocess.Popen("bash " + W + "/hb.sh", shell=True, start_new_session=True)

# ---- 8) 前台训练：崩溃自动 resume 重试，traceback 直接进批处理日志 ----
ENV = ("export PATH=/opt/bin:$PATH; export LD_LIBRARY_PATH=/usr/local/nvidia/lib64:$LD_LIBRARY_PATH; "
       "export MUJOCO_GL=egl; export HF_HOME=" + W + "/hf_cache;")
CMD = ("--policy.path=" + W + "/smolvla_base_fp32 --policy.push_to_hub=false "
       "--policy.input_features=null --policy.output_features=null "
       "--dataset.repo_id=lerobot/libero --dataset.revision=" + REV + " "
       "--batch_size=32 --num_workers=2 --accelerator.mixed_precision=fp16 "
       "--accelerator.gradient_accumulation.steps=1 --steps=8000 "
       "--policy.scheduler_warmup_steps=2000 --policy.scheduler_decay_steps=32000 "
       "--save_freq=2000 --seed=1000 "
       "--output_dir=" + W + "/outputs/m1_base_libero --job_name=m1_base_libero")
CKPTD = W + "/outputs/m1_base_libero/checkpoints"
MAX_TRY = 30
for attempt in range(1, MAX_TRY + 1):
    cks = sorted(int(d) for d in os.listdir(CKPTD) if d.isdigit()) if os.path.isdir(CKPTD) else []
    if not cks:
        shutil.rmtree(W + "/outputs/m1_base_libero", ignore_errors=True)  # 无 checkpoint 的遗留目录会触发 FileExistsError
    cmd = ENV + " torchrun --standalone --nproc-per-node=2 $(which lerobot-train) " + CMD + ((" --resume=true --config_path=" + CKPTD + "/%08d/pretrained_model" % cks[-1]) if cks else "")
    print("[8] 训练尝试 #%d (resume_ckpt=%s)" % (attempt, cks[-1] if cks else None), flush=True)
    ret = subprocess.call("bash -c 'set -o pipefail; " + cmd + " 2>&1 | tee -a " + W + "/logs/train.log'", shell=True)
    if ret == 0:
        beacon("DONE 8000updates ret=0")
        print("[8] 训练完成（8000 updates @ eff-bs64, DDP×2）", flush=True)
        break
    beacon("CRASH try%d ret=%d" % (attempt, ret))
    print("[8] 训练异常退出 ret=%d，30s 后 resume 重启" % ret, flush=True)
    time.sleep(30)
else:
    beacon("GIVEUP after 30 tries")
    raise RuntimeError("训练连续 %d 次失败，放弃（详见上方 train.log 输出）" % MAX_TRY)
print("ALL DONE", flush=True)
